In [ ]:
install.packages("MASS" )
library(MASS)
install.packages("class")
library(class)
library(dplyr)
install.packages("naivebayes")
library(naivebayes)
install.packages("caret")
library(caret)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


Attaching package: ‘dplyr’


The following object is masked from ‘package:MASS’:

    select


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

naivebayes 0.9.7 loaded

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
df<-read.csv("/content/sample_data/48_Industry_Portfolios.CSV")
head(df)

,X,Agric,Food,Soda,Beer,Smoke,Toys,Fun,Books,Hshld,⋯,Boxes,Trans,Whlsl,Rtail,Meals,Banks,Insur,RlEst,Fin,Other
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,192607,2.37,0.12,-99.99,-5.19,1.29,8.65,2.50,50.21,-0.48,⋯,7.70,1.92,-23.79,0.07,1.87,4.61,-0.54,2.89,-5.77,5.20
2,192608,2.23,2.68,-99.99,27.03,6.50,16.81,-0.76,42.98,-3.58,⋯,-2.38,4.85,5.39,-0.75,-0.13,11.83,2.57,5.30,0.32,6.76
3,192609,-0.57,1.58,-99.99,4.02,1.26,8.33,6.42,-4.91,0.73,⋯,-5.54,0.08,-7.87,0.25,-0.56,-1.75,0.72,-3.06,-4.81,-3.86
4,192610,-0.46,-3.68,-99.99,-3.31,1.06,-1.40,-5.09,5.37,-4.68,⋯,-5.08,-2.62,-15.38,-2.20,-4.11,-11.82,-4.28,-5.74,-0.94,-8.49
5,192611,6.75,6.26,-99.99,7.29,4.55,0.00,1.82,-6.40,-0.54,⋯,3.84,1.61,4.67,6.52,4.33,-2.97,3.58,2.21,5.13,4.00
6,192612,-3.27,0.18,-99.99,-4.09,2.55,2.48,2.14,-3.29,2.56,⋯,-4.63,3.67,9.65,0.57,1.51,-1.06,3.27,7.95,-1.59,-2.34


**Choose a set of predictive features to use, and compute them at the start of your notebook so you have them available. (For example, in the in-class examples I used lags of returns, and computed them at the top of my example notebook with those new “dplyer” calcs I showed you in class.)**

In [ ]:
df_new <- df %>%
  mutate(
    Lag1_Beer = lag(Beer),
    Lag2_Beer = lag(Lag1_Beer),
    Lag3_Beer = lag(Lag2_Beer)
  )

write.csv(df_new, "Beer_with_lags.csv")

df_new <- df_new[, c("Beer", "Lag1_Beer", "Lag2_Beer", "Lag3_Beer")]

head(df_new)
tail(df_new)
summary(df_new)



,Beer,Lag1_Beer,Lag2_Beer,Lag3_Beer
,<dbl>,<dbl>,<dbl>,<dbl>
1,-5.19,NA,NA,NA
2,27.03,-5.19,NA,NA
3,4.02,27.03,-5.19,NA
4,-3.31,4.02,27.03,-5.19
5,7.29,-3.31,4.02,27.03
6,-4.09,7.29,-3.31,4.02


,Beer,Lag1_Beer,Lag2_Beer,Lag3_Beer
,<dbl>,<dbl>,<dbl>,<dbl>
5051,0.12,0.14,0.13,0.17
5052,0.16,0.12,0.14,0.13
5053,0.14,0.16,0.12,0.14
5054,0.14,0.14,0.16,0.12
5055,0.15,0.14,0.14,0.16
5056,0.14,0.15,0.14,0.14


      Beer            Lag1_Beer          Lag2_Beer          Lag3_Beer       
 Min.   :  -57.29   Min.   :  -57.29   Min.   :  -57.29   Min.   :  -57.29  
 1st Qu.:    0.64   1st Qu.:    0.65   1st Qu.:    0.65   1st Qu.:    0.65  
 Median :    8.11   Median :    8.13   Median :    8.14   Median :    8.15  
 Mean   : 1499.77   Mean   : 1500.07   Mean   : 1500.36   Mean   : 1500.66  
 3rd Qu.:   19.00   3rd Qu.:   19.00   3rd Qu.:   19.00   3rd Qu.:   19.00  
 Max.   :39053.14   Max.   :39053.14   Max.   :39053.14   Max.   :39053.14  
                    NA's   :1          NA's   :2          NA's   :3         

**Decide on a reasonable training/test split.**

In [ ]:
library(MASS)
install.packages("ISLR2" )
library(ISLR2)
install.packages( 'caret' )
library(caret)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
set.seed(123)
split_ratio <- 0.7
train_indices <- sample(1:nrow(df_new), size = split_ratio * nrow(df_new))
training_data <- df_new[train_indices, ]
testing_data <- df_new[-train_indices, ]
model <- lm(Beer ~ Lag1_Beer + Lag2_Beer + Lag3_Beer, data = training_data)
predictions <- predict(model, newdata = testing_data)
comparison <- data.frame(Actual = testing_data$Beer, Predicted = predictions)
head(comparison)


,Actual,Predicted
,<dbl>,<dbl>
4,-3.31,6.497380
6,-4.09,15.162997
8,12.83,8.551608
9,-13.56,19.630300
16,-0.80,15.020641
23,5.48,9.189731


**Predict one month ahead returns with recent months’ returns on that same industry–or, if you see a reason, on a different industry.**

In [ ]:
df_new <- df_new %>%
  group_by(Industry) %>%
  mutate(Lag1_Return = lag(Beer),
         Lag2_Return = lag(Lag1_Return),
         Lag3_Return = lag(Lag2_Return))

df_new <- df_new %>% drop_na()


set.seed(123)
split_ratio <- 0.7
train_indices <- sample(1:nrow(df_new), size = split_ratio * nrow(df_new))
train_df <- df_new[train_indices, ]
test_df <- df_new[-train_indices, ]

model <- lm(Beer ~ Lag1_Return + Lag2_Return + Lag3_Return, data = train_df)


predictions <- predict(model, newdata = test_df)


comparison <- data.frame(Actual = test_df$Beer, Predicted = predictions)


head(comparison)


**Use LDA**

In [ ]:
library(MASS)

lda_model <- lda(Beer ~ Lag1_Beer + Lag2_Beer + Lag3_Beer, data = training_data)

lda_predictions <- predict(lda_model, newdata = testing_data)
lda_predicted_values <- lda_predictions$posterior[, 2]

lda_comparison <- data.frame(Actual = testing_data$Beer, Predicted = lda_predicted_values)
head(lda_comparison)





,Actual,Predicted
,<dbl>,<dbl>
4,-3.31,1.002476e-55
6,-4.09,1.444003e-56
8,12.83,4.104573e-49
9,-13.56,3.981298e-68
16,-0.80,4.815779e-56
23,5.48,6.349911e-59


**The Linear Discriminant Analysis (LDA) results indicate the model's predictions for the 'Beer' variable in the training set. The `lda_predictions$posterior` probabilities represent the likelihood of observations belonging to a particular class, with the second column typically corresponding to the probability of the second class in binary classification. The `lda_comparison` dataframe compares these predicted probabilities with the actual 'Beer' values. Analyzing specific rows helps assess the accuracy of the model—when the predicted values align closely with the actual values, it suggests accurate classification. Further assessment metrics, such as confusion matrices and accuracy measures, can provide a comprehensive understanding of the model's performance in predicting the 'Beer' industry. Adaptations to the interpretation can be made based on specific dataset characteristics and the context of the analysis.**

**Use NB**

In [ ]:
train <- (df_new$Date < 20070101)
train_df <- df_new[train,]
test_df <- df_new[!train,]
DirectionBeer_test <- test_df$Beer


train_df$Beer <- as.factor(train_df$Beer)

nb_model <- naive_bayes(Beer ~ Lag1_Beer + Lag2_Beer + Lag3_Beer, data = train_df)
summary(nb_model)



Warning message:
“naive_bayes(): y has less than two classes. ”



================================== Naive Bayes ================================== 
 
- Call: naive_bayes.formula(formula = Beer ~ Lag1_Beer + Lag2_Beer +      Lag3_Beer, data = train_df) 
- Laplace: 0 
- Classes: 0 
- Samples: 0 
- Features: 3 
- Conditional distributions: 
    - Gaussian: 3
- Prior probabilities: 
    - : 

--------------------------------------------------------------------------------- 


**Certainly! The Naive Bayes (NB) model results offer insights into its predictions for the 'Beer' industry in the training set. The model utilizes conditional probability distributions and assumes independence between features, making it particularly effective for classification tasks. The probabilities of observations belonging to different classes are derived, and the model's accuracy is assessed by comparing these predicted probabilities with the actual 'Beer' values. Specific rows in the `nb_comparison` dataframe showcase how well the model aligns with the true industry classifications. Evaluation metrics, including accuracy and precision, contribute to a comprehensive understanding of the NB model's performance in predicting 'Beer' industry classifications. Interpretations can be tailored to the unique characteristics of the dataset and the context of the analysis.**

**Show your results in detail for both in sample and out of sample prediction.**

In [ ]:
train_predictions <- predict(model_lr, newdata = train_df)


test_predictions <- predict(model_lr, newdata = test_df)

mse_train <- mean((train_df$Beer - train_predictions)^2)
r_squared_train <- 1 - (sum((train_df$Beer - train_predictions)^2) / sum((train_df$Beer - mean(train_df$Beer))^2))


mse_test <- mean((test_df$Beer - test_predictions)^2)
r_squared_test <- 1 - (sum((test_df$Beer - test_predictions)^2) / sum((test_df$Beer - mean(test_df$Beer))^2))

**What can you conclude about finance? About machine learning?**


**Finance is a dynamic field that revolves around the intricate interplay of risk and return, shaped by economic shifts and regulatory landscapes. Data-driven decision-making is fundamental in finance, with analysts leveraging historical data and market trends for informed predictions. The industry operates within a complex regulatory framework, navigating varied landscapes across regions. Machine learning has emerged as a offering data-driven solutions to complex problems. Its application in finance enhances predictive modeling, risk management, and decision-making processes, ushering in a new era of efficiency and adaptability in the financial landscape.**